## Execution Order

This notebook is stateful. Cells should generally be executed from top to bottom because later cells depend on variables created earlier (e.g., `document`, `chunks`, `collection`).

# 02 - Retrieval-Augmented Generation (RAG)

**Course:** FDP on Agentic AI & LLM Engineering

**Environment**
- Python 3.12
- VS Code
- Ollama
- LLM: qwen3:1.7b
- Vector Database: ChromaDB
- Embedding Model: Nomic Embed Text

---

## Objective

In this notebook, we build a complete Retrieval-Augmented Generation (RAG) pipeline using local LLMs and vector databases. By the end of this notebook, we will:

- Understand why RAG is needed
- Compare RAG with fine-tuning
- Build a local vector database
- Generate embeddings
- Perform semantic retrieval
- Generate grounded responses using retrieved context

This notebook serves as the foundation for Agentic AI systems, enterprise AI assistants, and robotics knowledge retrieval systems.

# Why Do We Need RAG?

Large Language Models possess extensive general knowledge but cannot access private, domain-specific, or recently updated information unless it is supplied during inference.

For example:

- A robotics company's maintenance manuals
- Internal software documentation
- Research papers published after the model's training cutoff
- Robot operating procedures
- Factory operating instructions

Without access to these documents, an LLM either states that it does not know the answer or generates an incorrect response (hallucination).

Retrieval-Augmented Generation (RAG) addresses this limitation by retrieving the most relevant documents and supplying them as additional context before the model generates its response.

                 Documents
                      │
                      ▼
                 Chunking
                      │
                      ▼
              Embedding Model
                      │
                      ▼
             Vector Database
                      ▲
                      │
               User Question
                      │
                      ▼
             Query Embedding
                      │
                      ▼
             Similarity Search
                      │
                      ▼
            Retrieved Chunks
                      │
                      ▼
               Prompt + Context
                      │
                      ▼
                    LLM
                      │
                      ▼
                  Final Answer

In [1]:
import chromadb

print(f"ChromaDB Version: {chromadb.__version__}")

ChromaDB Version: 1.5.9


In [2]:
import chromadb

# Create a persistent ChromaDB client
client = chromadb.PersistentClient(path="./chroma_db")

print("✅ ChromaDB initialized successfully.")

✅ ChromaDB initialized successfully.


In [3]:
collection = client.get_or_create_collection(
    name="dsu_aiml_curriculum"
)

print(f"Collection created: {collection.name}")

Collection created: dsu_aiml_curriculum


In [21]:
with open("knowledge_base/aiml_courses.txt", "r", encoding="utf-8") as file:
    document = file.read()

print(document)

Course: Reinforcement Learning

Semester: 7

Credits: 4

Description:
This course introduces intelligent agents that learn optimal behaviour by interacting with an environment. Topics include Markov Decision Processes, Dynamic Programming, Monte Carlo methods, Temporal Difference Learning, Q-Learning, SARSA and Deep Reinforcement Learning.

Applications:
Autonomous Robots
Game Playing
Industrial Automation
Self-driving Cars

---------------------------------------------------------

Course: Computer Vision

Semester: 6

Credits: 4

Description:
The course covers image processing, feature extraction, object detection, image classification, convolutional neural networks and visual perception for AI systems.

Applications:
Medical Imaging
Autonomous Vehicles
Surveillance
Robotics

---------------------------------------------------------

Course: Deep Learning

Semester: 6

Credits: 4

Description:
Students learn neural networks, CNNs, RNNs, Transformers and Generative AI models using PyT

In [9]:
from openai import OpenAI

# Connect to local Ollama
llm_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

print("Connected to Ollama")

Connected to Ollama


In [22]:
chunks = document.split("---------------------------------------------------------")

chunks = [chunk.strip() for chunk in chunks if chunk.strip()]

print(f"Number of Chunks: {len(chunks)}")

Number of Chunks: 4


In [23]:
embedding = llm_client.embeddings.create(
    model="nomic-embed-text",
    input=chunks[0]
)

print(type(embedding))

<class 'openai.types.create_embedding_response.CreateEmbeddingResponse'>


In [24]:
for i, chunk in enumerate(chunks):

    embedding = llm_client.embeddings.create(
        model="nomic-embed-text",
        input=chunk
    )

    collection.add(
        ids=[f"course_{i}"],
        documents=[chunk],
        embeddings=[embedding.data[0].embedding]
    )

print(f"✅ Stored {len(chunks)} course documents in ChromaDB.")

✅ Stored 4 course documents in ChromaDB.


In [25]:
print(collection.count())

4


In [31]:
query = "Which course teaches Quantum Teleportation?"

In [32]:

query_embedding = llm_client.embeddings.create(
    model="nomic-embed-text",
    input=query
)

results = collection.query(
    query_embeddings=[query_embedding.data[0].embedding],
    n_results=2
)

print(results["documents"])

[['Course: Reinforcement Learning\n\nSemester: 7\n\nCredits: 4\n\nDescription:\nThis course introduces intelligent agents that learn optimal behaviour by interacting with an environment. Topics include Markov Decision Processes, Dynamic Programming, Monte Carlo methods, Temporal Difference Learning, Q-Learning, SARSA and Deep Reinforcement Learning.\n\nApplications:\nAutonomous Robots\nGame Playing\nIndustrial Automation\nSelf-driving Cars', 'Course: Deep Learning\n\nSemester: 6\n\nCredits: 4\n\nDescription:\nStudents learn neural networks, CNNs, RNNs, Transformers and Generative AI models using PyTorch.\n\nApplications:\nNatural Language Processing\nComputer Vision\nSpeech Recognition\nGenerative AI']]


In [33]:
retrieved_context = "\n\n".join(results["documents"][0])

print(retrieved_context)

Course: Reinforcement Learning

Semester: 7

Credits: 4

Description:
This course introduces intelligent agents that learn optimal behaviour by interacting with an environment. Topics include Markov Decision Processes, Dynamic Programming, Monte Carlo methods, Temporal Difference Learning, Q-Learning, SARSA and Deep Reinforcement Learning.

Applications:
Autonomous Robots
Game Playing
Industrial Automation
Self-driving Cars

Course: Deep Learning

Semester: 6

Credits: 4

Description:
Students learn neural networks, CNNs, RNNs, Transformers and Generative AI models using PyTorch.

Applications:
Natural Language Processing
Computer Vision
Speech Recognition
Generative AI


In [34]:

prompt = f"""
You are the DSU AI/ML Academic Advisor.

Answer ONLY using the retrieved context.

If the answer is not present in the context,
reply with:
"I could not find this information in the curriculum."

Retrieved Context:
{retrieved_context}

Student Question:
{query}
"""

print(prompt)


You are the DSU AI/ML Academic Advisor.

Answer ONLY using the retrieved context.

If the answer is not present in the context,
reply with:
"I could not find this information in the curriculum."

Retrieved Context:
Course: Reinforcement Learning

Semester: 7

Credits: 4

Description:
This course introduces intelligent agents that learn optimal behaviour by interacting with an environment. Topics include Markov Decision Processes, Dynamic Programming, Monte Carlo methods, Temporal Difference Learning, Q-Learning, SARSA and Deep Reinforcement Learning.

Applications:
Autonomous Robots
Game Playing
Industrial Automation
Self-driving Cars

Course: Deep Learning

Semester: 6

Credits: 4

Description:
Students learn neural networks, CNNs, RNNs, Transformers and Generative AI models using PyTorch.

Applications:
Natural Language Processing
Computer Vision
Speech Recognition
Generative AI

Student Question:
Which course teaches Quantum Teleportation?



In [35]:
response = llm_client.chat.completions.create(
    model="qwen3:1.7b",
    messages=[
        {
            "role":"user",
            "content":prompt
        }
    ]
)

print(response.choices[0].message.content)

I could not find this information in the curriculum.


In [86]:
import importlib

import src.rag

importlib.reload(src.rag)

from src.rag import AcademicAdvisorRAG

In [87]:
advisor = AcademicAdvisorRAG()

advisor.ask(
    "Which course teaches intelligent agents and Q-learning?"
)

🔍 Retrieving relevant curriculum documents...
✓ Retrieved 3 relevant document(s).

Retrieved Documents:
1. Course: Reinforcement Learning
2. Course: Deep Learning
3. Course: AI for Space Agriculture
📝 Building prompt..
🤖 Generating response...
✓ Response generated.


'The course that teaches intelligent agents and Q-learning is the **Reinforcement Learning** course.'

In [88]:
from src.models import Course

course = Course(
    course_code="26AM2304",
    course_name="Artificial Intelligence",
    semester=3,
    credits=3,
    course_type="PCC",
    lecture=2,
    tutorial=0,
    practical=0,
    project=2
)

print(course)

Course(course_code='26AM2304', course_name='Artificial Intelligence', semester=3, credits=3, course_type='PCC', lecture=2, tutorial=0, practical=0, project=2)


In [64]:
import importlib

import src.rag
import src.parser
import src.models

importlib.reload(src.rag)
importlib.reload(src.parser)
importlib.reload(src.models)

from src.rag import AcademicAdvisorRAG
from src.parser import PDFParser
from src.models import Course

advisor = AcademicAdvisorRAG()


In [ ]:
parser = PDFParser(
    "knowledge_base/raw/syllabus.pdf"
)

courses = parser.extract_raw_courses(lines)

print(courses[0].course_name)
print(courses[0].course_code)

print("-" * 80)

PROBABILITY AND STATISTICS
26AM2301
--------------------------------------------------------------------------------
PROBABILITY AND STATISTICS
[As per Choice Based Credit System
(CBCS)scheme]
SEMESTER – III
Course Code
: 26AM2301
Credits
: 03
Hours /
Week
: 03 Hours
Total Hours
: 39 Hours
L–T–P–J
: 3–0–0–0
Course Learning Objectives:
This Course will enable students to:
1. Apply statistical principles and probability concepts to solve complex problems in real-
world scenarios involving uncertainty and randomness.
2. Evaluate and select appropriate probability distributions and statistical techniques to
analyze and interpret data accurately in various applications.
3. Justify the use of estimation methods and hypothesis testing techniques for drawing
meaningful inferences about population parameters.
4. Analyze and interpret sample test results for different statistical relationships, such as
means, variances, correlation coefficients, regression coefficients, goodness of fit, and
inde

In [63]:


advisor.reset_collection()
print(advisor.collection.count())
advisor.ingest_documents("knowledge_base/raw/syllabus.pdf")
print(advisor.collection.count())

Collection reset successfully.
0
Found 57 courses.
PROBABILITY AND STATISTICS -> 26AM2301
DATA STRUCTURES -> 26AM2302
PRINCIPLES OF DATABASE MANAGEMENT -> 26AM2303
ARTIFICIAL INTELLIGENCE -> 26AM2304
COMPUTER ORGANIZATION AND ARCHITECTURE -> 26AM2305
AI  FOR EMBEDDED SYSTEMS -> 22AM2306
JAVA PROGRAMMING -> 26AM2307
TRANSFORMS AND NUMERIAL TECHNIQUES -> 24AM2401
DESIGN AND ANALYSIS OF ALGORITHMS -> 26AM2402
COMPUTER NETWORKS -> 26AM2403
THEORY OF COMPUTATION and SYSTEM SOFTWARE -> 26AM2404
MACHINE LEARNING -> 26AM2405
FULL STACK DEVELOPMENT -> 26AM2406
SENSORS, SYSTEMS & IoT FUNDAMENTALS -> 26AM2407
AI FOR SUSTAINIBILITY -> 25A2408
DEEP LEARNING -> 26AM3501
SYSTEM SOFTWARE  AND OPERATING SYSTEM -> 25AM3502
NATURAL LANGUAGE MODELS -> 25AM3503
IMAGE PROCESSING AND COMPUTER VISION -> 25AM3504
INNOVATION AND ENTREPRENEURSHIP -> 26AM3505
(Skill Enhancement Course - III) -> 26AM3506
Accelerated Data Science -> 25AM3508
ETHICS, SAFETY & GOVERNANCE OF AGENTIC AI -> 25AM3509
Intelligent Speech P

In [65]:
advisor.ask("What is Artificial Intelligence?")

🔍 Retrieving relevant curriculum documents...
Distance : 0.7168516516685486
Course   : ARTIFICIAL INTELLIGENCE
Code     : 26AM2304
Distance : 0.7491580843925476
Course   : OPERATIONALIZING AGENTIC AI SYSTEMS
Code     : 25AM4709
Distance : 0.7907905578613281
Course   : AI FOR AUTONOMOUS VEHICLES & SMART MOBILITY
Code     : 25AM3610
✓ Retrieved 3 relevant document(s).

Retrieved Documents:
1. 
2. 
3. 
📝 Building prompt..
🤖 Generating response...
✓ Response generated.


'Artificial Intelligence (AI) is a branch of computer science that focuses on the development of intelligent systems capable of performing tasks that typically require human intelligence, such as problem-solving, decision-making, pattern recognition, and language understanding. In the context of this course, AI is applied to autonomous vehicles and smart mobility systems, leveraging machine learning (ML) and deep learning (DL) techniques to enable perception, prediction, localization, and decision-making. The course emphasizes the integration of AI with real-world transportation challenges, including safety, ethics, and sustainability, while addressing technical aspects like sensor fusion, neural networks, and autonomous navigation. It also highlights the importance of explainable AI (XAI) and ethical reasoning in ensuring responsible deployment of AI systems in mobility ecosystems.'